In [1]:
!pip install langchain langchain_chroma langchain-faiss langchain-classic faiss-cpu openai google-genai tiktoken langchain_openai  langchain_google-genai langchain-community wikipedia 

# Wikipedia Retriver

In [2]:
import wikipedia
from langchain_community.retrievers import WikipediaRetriever


/var/folders/z8/x7qpgkld3gg8x0_qgy81tmjc0000gn/T/ipykernel_2983/2657786518.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [3]:
retriever=WikipediaRetriever(top_k_results=2,lang="en")

In [4]:
query = "Strait of Hormuz"
try:
    # 3. Call invoke on the *instance* variable
    docs = retriever.invoke(query)
    
    for i, doc in enumerate(docs):
        print(f"Result {i+1}:")
        print(f"Title: {doc.metadata.get('title')}")
        print(doc.page_content[:300]) # Print a snippet of the content
        print("-" * 40)
        
except Exception as e:
    print(f"An error occurred: {e}")


Result 1:
Title: Strait of Hormuz
The Strait of Hormuz () is a waterway between the Persian Gulf and the Gulf of Oman. On the north coast lies Iran, and on the south coast lies the Musandam Peninsula under the Musandam Governorate of Oman, with a portion of the southwest of the peninsula under the United Arab Emirates (UAE). The str
----------------------------------------
Result 2:
Title: 2026 Strait of Hormuz crisis
Shipping traffic through the Strait of Hormuz, a major maritime choke point for world energy trade, has been largely blocked by Iran since 28 February 2026, when the United States and Israel launched an air war against Iran. In retaliation, the Iranian Revolutionary Guard Corps (IRGC) issued warning
----------------------------------------


# Vector store retriever

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

env_path=Path.cwd().parent.parent.parent /".env"
load_dotenv(env_path)

api_key=os.getenv("GEMINI_API_KEY")


if not api_key:
    print("API KEY MISSING")
    exit()
    
else:
    print("Initializing Model.....")


In [6]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [ ]:
embedding_model=GoogleGenerativeAIEmbeddings(model='gemini-embedding-001',api_key=api_key)

vector_store=Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection" 
)


In [ ]:
retriever=vector_store.as_retriever(search_kwargs={'k':2})

In [ ]:
query="What is chroma used for?"
results=retriever.invoke(query)

In [ ]:
for i,doc in enumerate(results):
    print(f"\n--Result {i+1} --")
    print(doc.page_content)


--Result 1 --
Chroma is a vector database optimized for LLM-based search.

--Result 2 --
Chroma is a vector database optimized for LLM-based search.


In [ ]:
results=vector_store.similarity_search(query,k=2)

In [ ]:
for i,doc in enumerate(results):
    print(f"\n--Result {i+1} --")
    print(doc.page_content)


--Result 1 --
Chroma is a vector database optimized for LLM-based search.

--Result 2 --
Chroma is a vector database optimized for LLM-based search.


# MMR

In [ ]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [ ]:
from langchain_community.vectorstores import FAISS
embedding_model=GoogleGenerativeAIEmbeddings(model='gemini-embedding-2',api_key=api_key)

vector_store=FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [ ]:
retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={'k':3,"lambda_mult":1}
)

In [ ]:
query="What is langchain"
results=retriever.invoke('query')
print(results)

[Document(id='ab098645-8880-4803-b132-b4ee71544db3', metadata={}, page_content='Embeddings are vector representations of text.'), Document(id='85d9dd83-b251-41a5-a9d3-1d1a45c10991', metadata={}, page_content='MMR helps you get diverse results when doing similarity search.'), Document(id='83f999e1-c0b3-4026-a6a1-5bb97ab8dba1', metadata={}, page_content='Chroma is used to store and search document embeddings.')]


In [ ]:
for i,doc in enumerate(results):
    print(f"\n--Result {i+1} --")
    print(doc.page_content)


--Result 1 --
Embeddings are vector representations of text.

--Result 2 --
MMR helps you get diverse results when doing similarity search.

--Result 3 --
Chroma is used to store and search document embeddings.


# Multiquery Retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_classic.retrievers.multi_query import MultiQueryRetriever


In [ ]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [ ]:
embedding_model=GoogleGenerativeAIEmbeddings(model='gemini-embedding-2',api_key=api_key)
vector_store=FAISS.from_documents(documents=all_docs,embedding=embedding_model)

In [ ]:
similarity_retriver=vector_store.as_retriever(search_type='similarity',search_kwargs={'k':5})

In [ ]:
multi_query_retriver=MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={'k':5}),
    llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
)

In [ ]:
query="How to improve energy levels and maintain balance?"


In [ ]:
similarity_results=similarity_retriver.invoke(query)
multiquery_results=multi_query_retriver.invoke(query)

In [ ]:
for i,doc in enumerate(similarity_results):
    print(f"\n--Result {i+1} --")
    print(f"{doc.page_content}")
    
print("*"*150)

for i,doc in enumerate(multiquery_results):
    print(f"\n--Result {i+1} --")
    print(doc.page_content)


--Result 1 --
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--Result 2 --
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--Result 3 --
Consuming leafy greens and fruits helps detox the body and improve longevity.

--Result 4 --
Deep sleep is crucial for cellular repair and emotional regulation.

--Result 5 --
Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************************************************************************************************************

--Result 1 --
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--Result 2 --
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--Result 3 --
Deep sleep is crucial for cellular repair and emotional regulation.

--Result 4 --
Consuming leafy greens and fruits helps detox the body and improve longevity.

--Result 5 --
Regula